# Étape 0 : Définir le problème
## Projet : Analyser les transactions pour détecter la fraude bancaire

**Pourquoi cette étape ?** Avant d'écrire la moindre ligne de code d'analyse, un data scientist doit comprendre *quel problème il résout*, *pour qui*, et *comment savoir s'il a réussi*. 

## 1. Contexte du projet

Une banque traite chaque jour des millions de transactions par carte bancaire. Une petite fraction est frauduleuse (carte volée, piratage, usurpation d'identité...).

**Constat clé :** la fraude est un phénomène **rare** mais **coûteux**.

| Élément | Description |
|---|---|
| Acteur | Une banque / un service de détection de fraude |
| Donnée disponible | Historique de transactions étiquetées (fraude ou non) |
| Type de problème | Classification binaire supervisée |
| Difficulté principale | Classes très déséquilibrées |

## 2. Objectif métier

### 2.1 Formuler l'objectif 

> *« Détecter automatiquement au moins 80 % des transactions frauduleuses tout en limitant à moins de 1 % le taux de transactions légitimes bloquées à tort. »*


### 2.2 Temps réel ou analyse a posteriori ?

| Approche | Description | Contrainte |
|---|---|---|
| **Temps réel** | Le modèle décide en quelques millisecondes d'accepter ou bloquer la transaction | Modèle rapide, simple à déployer |


## 3. Type de problème de machine learning

- **Apprentissage supervisé** : on dispose d'exemples déjà étiquetés (`Class` = 0 ou 1).
- **Classification binaire** : la sortie est une catégorie parmi deux.
  - `0` = transaction normale
  - `1` = transaction frauduleuse
- **Alternative possible** : la *détection d'anomalies* (non supervisée), utile quand on n'a pas d'étiquettes. Ici, on a des étiquettes, donc on reste sur la classification supervisée.

## 4. Le coût des erreurs (point le plus important)

Un modèle peut se tromper de deux façons :

| | Prédit : normal | Prédit : fraude |
|---|---|---|
| **Réalité : normal** | ✅ Vrai négatif | ⚠️ **Faux positif** : client légitime bloqué |
| **Réalité : fraude** | ❌ **Faux négatif** : fraude non détectée | ✅ Vrai positif |

Ces deux erreurs **n'ont pas le même coût** :
- **Faux négatif** : la banque perd le montant de la fraude (et parfois doit rembourser le client).
- **Faux positif** : le client est mécontent, appelle le support, risque de changer de banque.

### Simulation : comparer deux stratégies

Imaginons 100 000 transactions dont 200 fraudes (0,2 %). Comparons :
- **Modèle A** : prudent, détecte peu de fraudes mais bloque peu de clients
- **Modèle B** : agressif, détecte beaucoup de fraudes mais bloque plus de clients légitimes

**À retenir :** le meilleur modèle n'est pas forcément celui qui a la meilleure « précision » globale, mais celui qui **minimise le coût métier total**. Cela guidera le choix de nos métriques à l'étape 8.

## 5. Choix des métriques de réussite

### Pourquoi l'accuracy est un piège ici

Avec 0,2 % de fraudes, un modèle qui répond **toujours « pas de fraude »** obtient 99,8 % d'accuracy... tout en étant totalement inutile. Démonstration :

In [3]:
n_total = 100_000
n_fraudes = 200

# Un "modèle" idiot qui prédit toujours "normal"
accuracy_modele_idiot = (n_total - n_fraudes) / n_total
fraudes_detectees = 0

print(f"Accuracy du modèle idiot : {accuracy_modele_idiot:.2%}")
print(f"Fraudes détectées        : {fraudes_detectees} sur {n_fraudes}")

Accuracy du modèle idiot : 99.80%
Fraudes détectées        : 0 sur 200


### Les métriques à utiliser à la place

| Métrique | Question à laquelle elle répond | Formule simplifiée |
|---|---|---|
| **Rappel (Recall)** | Parmi les vraies fraudes, combien en ai-je détecté ? | VP / (VP + FN) |
| **Précision (Precision)** | Parmi mes alertes, combien sont de vraies fraudes ? | VP / (VP + FP) |
| **F1-score** | Compromis entre précision et rappel | moyenne harmonique des deux |
| **AUC-PR** | Qualité globale sur données déséquilibrées | aire sous la courbe Précision-Rappel |
| **AUC-ROC** | Capacité à séparer les deux classes | aire sous la courbe ROC |


## 6. Les données disponibles

Dataset recommandé pour débuter : **Credit Card Fraud Detection** (Kaggle, ULB Machine Learning Group).

| Caractéristique | Valeur |
|---|---|
| Nombre de transactions | 284 807 |
| Nombre de fraudes | 492 (≈ 0,17 %) |
| Période | 2 jours (septembre 2013, cartes européennes) |
| Variables `V1` à `V28` | Composantes issues d'une PCA (anonymisées pour confidentialité) |
| `Time` | Secondes écoulées depuis la première transaction |
| `Amount` | Montant de la transaction |
| `Class` | Cible : 0 = normal, 1 = fraude |

**Lien :** https://www.kaggle.com/mlg-ulb/creditcardfraud

**Limites à connaître dès maintenant :**
- Les variables V1-V28 sont anonymisées : impossible d'interpréter leur signification métier.
- Seulement 2 jours de données : le modèle ne capture pas de saisonnalité.
- Données de 2013 : les techniques de fraude ont évolué depuis.

## 7. Contraintes et risques


| Catégorie | Exemples de questions | Ma réponse |
|---|---|---|
| **Technique** | Le modèle doit-il répondre en moins de 100 ms ? | |
| **Réglementaire** | Doit-on pouvoir expliquer pourquoi une transaction est bloquée ? | |
| **Éthique / biais** | Le modèle risque-t-il de pénaliser certaines catégories de clients ? | |
| **Données** | Les données sont-elles représentatives de la réalité actuelle ? | |
| **Évolution** | Les fraudeurs adaptent leurs méthodes : comment garder le modèle à jour ? | |

## 8. Critères de succès du projet

Définir à l'avance ce qui sera considéré comme un succès, **avant** de voir les résultats. Cela évite de biaiser l'évaluation après coup.


In [5]:
criteres_succes = {
    "recall_minimum": 0.80,        # au moins 80 % des fraudes détectées
    "precision_minimum": 0.50,     # au moins 50 % des alertes sont de vraies fraudes
    "auc_pr_minimum": 0.70,        # qualité globale sur données déséquilibrées
}

for critere, seuil in criteres_succes.items():
    print(f"{critere:<20} >= {seuil}")

recall_minimum       >= 0.8
precision_minimum    >= 0.5
auc_pr_minimum       >= 0.7


## 9. Plan du projet

| # | Étape | Livrable |
|---|---|---|
| 0 | Définir le problème | ✅ Ce notebook |
| 1 | Collecte des données | Fichier `creditcard.csv` chargé |
| 2 | Exploration (EDA) | Graphiques et statistiques |
| 3 | Nettoyage | Dataset propre |
| 4 | Feature engineering | Variables prêtes pour le modèle |
| 5 | Séparation train / test | Jeux d'entraînement et de test |
| 6 | Gestion du déséquilibre | Données d'entraînement rééquilibrées |
| 7 | Entraînement des modèles | Modèles entraînés |
| 8 | Évaluation | Métriques comparées |
| 9 | Optimisation | Hyperparamètres ajustés |
| 10 | Interprétation | Variables importantes identifiées |
| 11 | Conclusion | Rapport final |
| 12 | Déploiement (bonus) | API ou tableau de bord |